# Pipetly - API Clients & Orchestrator Test Notebook

Use this notebook to interactively run searches against any combination of the
configured API clients and inspect the raw papers returned before filtering.

**Sections**
1. Setup & imports
2. Configuration - set your query here
3. Individual client search (EuropePMC - SemanticScholar - Elsevier - CrossRef - OpenAlex - Scopus - PMC)
4. Deduplication - combine & deduplicate all search results by DOI
5. Individual client full-text retrieval (EuropePMC - PMC - Elsevier - Semantic Scholar - Unpaywall)
6. Parse & display results
7. Analysis (counts by source, full-text availability, year distribution)

## 1 · Setup & Imports

In [ ]:
import sys, os, asyncio, logging, warnings
from pathlib import Path

# -- Make sure the project root is on the path --------------------------------
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

# -- Standard imports -----------------------------------------------------------
import pandas as pd
import json
warnings.filterwarnings("ignore")

# -- Logging - show INFO from Pipetly modules ----------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("notebook")

# -- Pipetly imports ------------------------------------------------------------
from config import get_settings
from models.paper import Paper, FullText
from models.query import ExpandedQuery
from api_clients import (
    EuropePMCClient,
    SemanticScholarClient,
    ElsevierClient,
    CrossRefClient,
    OpenAlexClient,
    ScopusClient,
    PMCClient,
    UnpaywallClient,
)
from processors.orchestrator import MultiSourceOrchestrator

settings = get_settings()
print(f"Model            : {settings.gemini_model}")
print(f"Max/source       : {settings.max_papers_per_source}")
print(f"Elsevier key set : {bool(settings.elsevier_api_key)}")
print(f"S2 key set       : {bool(settings.semantic_scholar_api_key)}")
print(f"Unpaywall email  : {bool(settings.unpaywall_email)}")

## 2 · Configuration

Edit the cell below to set your search query and per-source limit.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
QUERY = "FmtA structure"   # ← change to any search string
MAX_PER_SOURCE = 10                      # papers to fetch per API call

# ── Helper: convert a list of Paper objects to a tidy DataFrame ──────────────
def papers_to_df(papers: list[Paper]) -> pd.DataFrame:
    rows = []
    for p in papers:
        rows.append({
            "title":          p.title,
            "doi":            p.doi or "",
            "year":           p.year,
            "source":         p.source,
            "authors":        "; ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else ""),
            "has_full_text":  p.full_text is not None,
            "abstract_only":  getattr(p.full_text, "is_abstract_only", None),
            "url":            p.url or "",
            "abstract":       (p.abstract or "")[:200],
        })
    return pd.DataFrame(rows)

print("Config ready. QUERY =", repr(QUERY))

## 3 · Individual Client Search

Run each subsection independently to inspect what a single API returns.

### 3a · Europe PMC

In [ ]:
async with EuropePMCClient() as client:
    epmc_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"EuropePMC returned {len(epmc_papers)} papers")
papers_to_df(epmc_papers)

### 3b · Semantic Scholar

In [ ]:
async with SemanticScholarClient() as client:
    s2_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"Semantic Scholar returned {len(s2_papers)} papers")
papers_to_df(s2_papers)

### 3c · Elsevier (ScienceDirect)

In [ ]:
async with ElsevierClient() as client:
    els_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"Elsevier returned {len(els_papers)} papers")
papers_to_df(els_papers)

### 3d · CrossRef

In [ ]:
async with CrossRefClient() as client:
    cr_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"CrossRef returned {len(cr_papers)} papers")
papers_to_df(cr_papers)

### 3e · OpenAlex

In [ ]:
async with OpenAlexClient() as client:
    oa_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"OpenAlex returned {len(oa_papers)} papers")
papers_to_df(oa_papers)

### 3f · Scopus

In [ ]:
async with ScopusClient() as client:
    scopus_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"Scopus returned {len(scopus_papers)} papers")
papers_to_df(scopus_papers)

### 3g · PubMed / PMC

In [ ]:
async with PMCClient() as client:
    pmc_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"PMC returned {len(pmc_papers)} papers")
papers_to_df(pmc_papers)

## 4 - Deduplication

Combines results from all section-3 clients, deduplicates by DOI (falling
back to normalised title for records without a DOI), and produces `_pool` - the
unique paper list used by every full-text retrieval cell in section 5.

In [ ]:
from collections import defaultdict

# -- 1. All raw results per source ---------------------------------------------
_source_raw: dict[str, list] = {
    "europe_pmc":       epmc_papers,
    "semantic_scholar": s2_papers,
    "elsevier":         els_papers,
    "crossref":         cr_papers,
    "openalex":         oa_papers,
    "scopus":           scopus_papers,
    "pmc":              pmc_papers,
    "unpaywall":        [],            # no search endpoint
}

_pool_raw: list = []
for _papers in _source_raw.values():
    _pool_raw.extend(_papers)

# -- 2. Deduplicate (DOI-first, normalised title fallback) ---------------------
_pool_seen: dict[str, "Paper"] = {}   # uid -> first-seen Paper object
_uid_sources: dict[str, list[str]] = defaultdict(list)  # uid -> [sources]

for _src, _papers in _source_raw.items():
    for _p in _papers:
        _uid = _p.unique_id()
        _uid_sources[_uid].append(_src)
        if _uid not in _pool_seen:
            _pool_seen[_uid] = _p

_pool = list(_pool_seen.values())

# -- 3. Deduplication summary ---------------------------------------------------
_n_raw = len(_pool_raw)
_n_uniq = len(_pool)
_n_dupes = _n_raw - _n_uniq
_shared = {uid: srcs for uid, srcs in _uid_sources.items() if len(srcs) > 1}

print("=" * 55)
print("  DEDUPLICATION SUMMARY")
print("=" * 55)
print(f"  Raw papers (all sources combined) : {_n_raw:>5}")
print(f"  Unique papers after dedup         : {_n_uniq:>5}")
print(f"  Duplicate records removed         : {_n_dupes:>5}")
print(f"  Papers shared by >=2 sources      : {len(_shared):>5}")
print()

# -- 4. Per-source breakdown ----------------------------------------------------
print(f"{'Source':<22} {'Raw':>5}  {'w/ DOI':>7}  {'No DOI':>7}")
print("-" * 46)
for _src, _papers in _source_raw.items():
    _n_doi = sum(1 for _p in _papers if _p.doi)
    _n_none = len(_papers) - _n_doi
    print(f"  {_src:<20} {len(_papers):>5}  {_n_doi:>7}  {_n_none:>7}")
print("-" * 46)
print(f"  {'TOTAL':<20} {_n_raw:>5}  {sum(1 for _p in _pool_raw if _p.doi):>7}  {sum(1 for _p in _pool_raw if not _p.doi):>7}")
print()

# -- 5. Cross-source overlap ----------------------------------------------------
if _shared:
    print("Cross-source duplicates (sample of up to 5):")
    for _uid, _srcs in list(_shared.items())[:5]:
        print(f"  {_uid[:52]}  <- {', '.join(_srcs)}")
else:
    print("No cross-source duplicates found in this query.")

In [ ]:
# ── Deduplication analytics as a DataFrame ────────────────────────────────────
import pandas as pd

_dedup_rows = []
for _src, _papers in _source_raw.items():
    _with_doi  = sum(1 for _p in _papers if _p.doi)
    _no_doi    = len(_papers) - _with_doi
    # How many of this source's papers are unique vs. seen earlier
    _src_uids  = {(_p.doi.strip().lower() if _p.doi else _p.title.strip().lower())
                  for _p in _papers}
    _unique_contributed = sum(
        1 for uid in _src_uids
        if _uid_sources[uid][0] == _src   # this source was first to see it
    )
    _dedup_rows.append({
        "source":             _src,
        "raw_papers":         len(_papers),
        "with_doi":           _with_doi,
        "no_doi":             _no_doi,
        "unique_contributed": _unique_contributed,
        "duplicated_away":    len(_papers) - _unique_contributed,
    })

_dedup_df = pd.DataFrame(_dedup_rows).set_index("source")
print(f"Pool size after dedup: {len(_pool)} unique papers\n")
_dedup_df

## 5 - Individual Client Full-Text Retrieval

Each subsection runs `fetch_full_text` for **every paper in `_pool`**
(built and deduplicated in section 4) through that one client.
This tests each API's full-text coverage independently of which client
originally discovered a paper.

Results are stored as `{client}_ft_map: dict[str, str]` (unique-id -> text) and
merged in **5j** to build `raw_papers` for sections 6 and 7.

In [ ]:
# ── 5·0  Verify the deduplicated pool is available ───────────────────────────
# `_pool` and `_pool_seen` are built in section 4.
# Run all section-3 cells AND the section-4 deduplication cells first.
assert "_pool" in dir(), "Run section 4 deduplication cells before section 5."
print(f"Pool ready: {len(_pool)} unique papers — running full-text clients below.")

### 5a · Europe PMC

In [ ]:
epmc_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with EuropePMCClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            epmc_ft_map[_uid] = _ft

print(f"EuropePMC: full text retrieved for {len(epmc_ft_map)} / {len(_pool)} papers")
for _uid, _ft_obj in list(epmc_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


### 5b · Semantic Scholar

In [ ]:
s2_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with SemanticScholarClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            s2_ft_map[_uid] = _ft

print(f"Semantic Scholar: full text retrieved for {len(s2_ft_map)} / {len(_pool)} papers")
for _uid, _ft_obj in list(s2_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


### 5c · Elsevier (ScienceDirect)

In [ ]:
els_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with ElsevierClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            els_ft_map[_uid] = _ft

print(f"Elsevier: full text retrieved for {len(els_ft_map)} / {len(_pool)} papers")
if not els_ft_map:
    print("  (0 results is normal if ELSEVIER_API_KEY is not set or articles are not licensed)")
for _uid, _ft_obj in list(els_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


### 5d · PubMed / PMC

In [ ]:
pmc_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with PMCClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            pmc_ft_map[_uid] = _ft

print(f"PMC: full text retrieved for {len(pmc_ft_map)} / {len(_pool)} papers")
if not pmc_ft_map:
    print("  (0 results means no paper in the pool has a PMC open-access record)")
for _uid, _ft_obj in list(pmc_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


### 5e · Unpaywall

No search endpoint — resolves any DOI to the best open-access PDF.  
Requires `UNPAYWALL_EMAIL` in `.env`.

In [ ]:
upw_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with UnpaywallClient() as client:
    for _p in _pool:
        if not _p.doi:
            continue   # Unpaywall requires a DOI
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            upw_ft_map[_uid] = _ft

_doi_count = sum(1 for _p in _pool if _p.doi)
print(f"Unpaywall: full text retrieved for {len(upw_ft_map)} / {_doi_count} papers with a DOI")
if not upw_ft_map:
    print("  (0 results is normal for subscription-only papers or if UNPAYWALL_EMAIL is not set)")
for _uid, _ft_obj in list(upw_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


### 5j - Consolidate - attach best full text & build `raw_papers`

Merges the candidate pool with the per-client full-text maps from 5a-5e.
For each unique paper the **first** client that returned text wins (priority order:
EuropePMC -> PMC -> Elsevier -> Semantic Scholar -> Unpaywall).
The resulting `raw_papers` list is used for sections 6 and 7.

In [ ]:
from models.paper import FullText, FullTextFormat

# -- Full-text maps from sections 5a-5e (uid -> FullText) ----------------------
# Priority: first map that has a hit for a given paper wins.
# Order must match FullTextRetriever._CLIENT_ORDER in processors/full_text_retriever.py
_ft_maps_ordered = [
    ("EuropePMC",        epmc_ft_map),
    ("PMC",              pmc_ft_map),
    ("Elsevier",         els_ft_map),
    ("Semantic Scholar", s2_ft_map),
    ("Unpaywall",        upw_ft_map),
]

def _attach_ft(paper: "Paper", ft: FullText) -> None:
    """Attach a FullText object to a paper."""
    paper.full_text = ft

# raw_papers = deduplicated pool (already built above as _pool / _pool_seen)
raw_papers = list(_pool_seen.values())

attached = 0
source_counts: dict[str, int] = {}
for _p in raw_papers:
    _uid = _p.unique_id()
    for _source, _ft_map in _ft_maps_ordered:
        if _uid in _ft_map:
            _ft_obj = _ft_map[_uid]
            _attach_ft(_p, _ft_obj)
            _p.ft_retrieved_by = {
                "EuropePMC": "europe_pmc",
                "PMC": "pmc",
                "Elsevier": "elsevier",
                "Semantic Scholar": "semantic_scholar",
                "Unpaywall": "unpaywall",
            }.get(_source, _source.lower())
            source_counts[_source] = source_counts.get(_source, 0) + 1
            attached += 1
            break   # first hit wins

print(f"Full text attached to {attached} / {len(raw_papers)} papers")
print("\nBreakdown by supplying client:")
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {src:<22} {cnt:>4} papers")

## 6 · Parse & Display Results

Convert `raw_papers` (built in section 5j) to a tidy DataFrame and display it.

In [ ]:
df = papers_to_df(raw_papers)

# Display settings
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

print(f"Shape: {df.shape[0]} papers × {df.shape[1]} columns\n")
df[["source", "title", "doi", "year", "has_full_text", "abstract_only"]]

## 7 · Analysis

Counts by source, full-text availability, year distribution, DOI coverage,
and a per-source full-text retrieval summary.

In [ ]:
# ── 7a · Papers per source & full-text availability ──────────────────────────
print("=== Papers per source ===")
print(df.groupby("source").size().sort_values(ascending=False).to_string())

print("\n=== Full-text availability ===")
print(df.groupby("source")["has_full_text"].value_counts().to_string())

print("\n=== Abstract-only papers ===")
print(df["abstract_only"].value_counts().to_string())

print("\n=== Papers with a DOI ===")
has_doi = df["doi"].str.strip().astype(bool)
print(f"  With DOI    : {has_doi.sum()}")
print(f"  Without DOI : {(~has_doi).sum()}")

In [ ]:
# ── 7b · Year distribution & unique DOI count ────────────────────────────────
print("=== Publication year distribution ===")
year_counts = df["year"].dropna().astype(int).value_counts().sort_index(ascending=False)
print(year_counts.to_string())

# ── Deduplicated DOI count (simulating FilterPipeline step 1) ────────────────
print("\n=== Unique DOIs (dedup preview) ===")
unique_dois = df[df["doi"].str.strip().astype(bool)]["doi"].nunique()
print(f"  Unique DOIs : {unique_dois} / {has_doi.sum()} total DOI papers")

### 7c · Full-Text Retrieval Success per Client

Shows which clients were able to return full text in section 5.

In [ ]:
_ft_summary = {
    "EuropePMC":        epmc_ft_map,
    "PMC":              pmc_ft_map,
    "Elsevier":         els_ft_map,
    "Semantic Scholar": s2_ft_map,
    "Unpaywall":        upw_ft_map,
}

_pool_size = len(_pool)
print("=== Full-Text Retrieval Summary (all clients vs. full pool) ===")
print(f"{'Client':<22} {'Papers with FT':>15}  {'Coverage':>10}")
print("-" * 52)
for name, ft_map in _ft_summary.items():
    n = len(ft_map)
    pct = n / _pool_size * 100 if _pool_size else 0
    bar = "#" * int(pct / 5)
    print(f"{name:<22} {n:>8} / {_pool_size:<5}  {pct:5.1f}%  {bar}")

# Union coverage: how many papers got full text from at least one client
_covered = set()
for ft_map in _ft_summary.values():
    _covered |= set(ft_map.keys())
print(f"\nUnion coverage : {len(_covered)} / {_pool_size} papers "
      f"({len(_covered)/max(_pool_size,1)*100:.1f}%) have full text from >=1 client")

### 7d · Source Overlap — DOI Coverage per Client

Shows how many unique DOIs each client contributed and how many were shared
across multiple sources (indicating duplicates that were collapsed).

In [ ]:
from collections import defaultdict

_source_raw = {
    "europe_pmc":       epmc_papers,
    "semantic_scholar": s2_papers,
    "elsevier":         els_papers,
    "crossref":         cr_papers,
    "openalex":         oa_papers,
    "scopus":           scopus_papers,
    "pmc":              pmc_papers,
    "unpaywall":        [],            # no search results
}

# Build DOI -> sources mapping from raw search results
_doi_sources: dict[str, list[str]] = defaultdict(list)
for _src, _papers in _source_raw.items():
    for _p in _papers:
        if _p.doi:
            _doi_sources[_p.doi.strip().lower()].append(_src)

# Per-source stats
print("=== Raw search results per source ===")
print(f"{'Source':<22} {'Papers':>8} {'w/ DOI':>8} {'Unique DOIs':>12}")
print("-" * 54)
for _src, _papers in _source_raw.items():
    n_doi = sum(1 for _p in _papers if _p.doi)
    unique = len({_p.doi.strip().lower() for _p in _papers if _p.doi})
    print(f"{_src:<22} {len(_papers):>8} {n_doi:>8} {unique:>12}")

# Cross-source overlap
_shared = {doi: srcs for doi, srcs in _doi_sources.items() if len(srcs) > 1}
print("\n=== Cross-source DOI overlap ===")
print(f"DOIs found by only 1 source  : {sum(1 for s in _doi_sources.values() if len(s)==1)}")
print(f"DOIs shared by 2+ sources    : {len(_shared)}")
if _shared:
    print("\nShared DOIs (sample):")
    for doi, srcs in list(_shared.items())[:5]:
        print(f"  {doi[:50]}  <- {', '.join(srcs)}")